# Notebook 04 — The LSTM autoencoder

*Corrected version of the original `03_lstm_autoencoder.ipynb`.*

**What this notebook does.** It builds the LSTM autoencoder used everywhere in the project,
checks its size and shapes, and runs a short sanity training on one meta-training channel.

**Why.** It confirms the model is exactly the one described in the manuscript, and shows
what a fair sanity check looks like.

**Input.** The raw SMAP release. **Output.** `lstm_ae_init_seed42.pt` and one figure.

### What was corrected

1. **The model definition now lives in `maml_common.py`** and is imported, not re-typed.
   The architecture is unchanged: 69,481 parameters at 25 features (checked by an assert).
2. **The untrained-model check was misread.** The original printed clearly different errors
   for anomaly and normal windows from an *untrained* model (0.0415 vs 0.0256) and called
   them "similar". An untrained model has learned nothing about anomalies, so any such
   difference comes from the windows themselves: which file they are cut from, and their
   amplitude. The corrected check adds normal windows from the test file to separate the
   two effects.
3. **The separation check used training windows.** The original compared anomaly windows
   against the same normal windows the model had just been trained on. The corrected check
   holds out the last 20% of the normal windows.
4. **The saved starting weights were not reproducible.** The original re-created the model
   after the sanity training without resetting the seed. The seed is now set right before.
5. **Warning added:** the saved file is an untrained starting point. In the original
   pipeline, the original notebook 05 loaded this file as its "Static-AE" baseline, so that baseline was
   an untrained network.

In [1]:
import os, sys

def _find_common():
    """Find maml_common.py: this folder when run locally, /kaggle/input on Kaggle."""
    for root in [os.getcwd(), "/kaggle/input"]:
        if os.path.isdir(root):
            for d, _, files in os.walk(root):
                if "maml_common.py" in files:
                    return d
    raise FileNotFoundError("maml_common.py not found. Run from the Corrected-Notebooks folder, "
                            "or attach that folder to Kaggle as a Dataset.")

sys.path.insert(0, _find_common())
import maml_common as sc
SMOKE = os.environ.get("SMAP_SMOKE") == "1"     # tiny settings for testing only
OUT = sc.output_dir(smoke=SMOKE)
print("shared code:", sc.__file__)
print("outputs go to:", OUT)
print("code version:", sc.git_commit())

import numpy as np, torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sc.seed_everything(42)
print("device:", DEVICE)

shared code: /home/user/Objective-2/Corrected-Notebooks/maml_common.py
outputs go to: /home/user/Objective-2/Corrected-Notebooks/outputs
code version: 85c9644f9e8f5ee6eae4e65bf032fc8577223ed3


device: cpu


## 1 — Build the model and count parameters

In [2]:
model = sc.LSTMAutoencoder(input_size=25).to(DEVICE)
print(model)
n_params = sc.count_params(model)
print(f"trainable parameters: {n_params:,}")
assert n_params == 69481

LSTMAutoencoder(
  (encoder): LSTMEncoder(
    (lstm1): LSTM(25, 64, batch_first=True)
    (lstm2): LSTM(64, 32, batch_first=True)
    (fc): Linear(in_features=32, out_features=16, bias=True)
  )
  (decoder): LSTMDecoder(
    (lstm1): LSTM(16, 32, batch_first=True)
    (lstm2): LSTM(32, 64, batch_first=True)
    (fc): Linear(in_features=64, out_features=25, bias=True)
  )
)
trainable parameters: 69,481


## 2 — Trace one batch through the model

In [3]:
x = torch.randn(8, 30, 25, device=DEVICE)
with torch.no_grad():
    h1, _ = model.encoder.lstm1(x)
    _, (h2, _) = model.encoder.lstm2(h1)
    z = model.encoder.fc(h2[-1])
    xh = model(x)
    s = model.reconstruction_error(x)
for name, t in [("input", x), ("encoder LSTM 1", h1), ("encoder LSTM 2, last state", h2[-1]),
                ("code", z), ("reconstruction", xh), ("scores (one per window)", s)]:
    print(f"{name:28s} {tuple(t.shape)}")
assert xh.shape == x.shape and s.shape == (8,)

input                        (8, 30, 25)
encoder LSTM 1               (8, 30, 64)
encoder LSTM 2, last state   (8, 32)
code                         (8, 16)
reconstruction               (8, 30, 25)
scores (one per window)      (8,)


## 3 — An untrained model on real data

We use P-3, a meta-training channel (not an evaluation channel). Three groups of windows
are compared:

- normal windows from the **training file**,
- normal windows from the **test file** (windows with no labelled anomalous timestep),
- anomaly windows from the test file, built as in the original.

An untrained network knows nothing about anomalies. If it still scores the two test-file
groups differently from the training-file group, the difference comes from the files, not
from the anomalies.

In [4]:
data, labels = sc.build_smap_dataset(["P-3"])
d = data["P-3"]
test_w = sc.create_windows(d["test"], 30, 10)
starts = np.arange(len(test_w)) * 10
covered = np.array([d["labels"][s:s + 30].any() for s in starts])
groups = {"normal, training file": d["normal_windows"],
          "normal, test file": test_w[~covered],
          "anomaly windows, test file": d["legacy_anomaly_windows"]}
torch.manual_seed(42)
untrained = sc.LSTMAutoencoder(25).to(DEVICE)
for name, w in groups.items():
    e = sc.window_errors(untrained, w, DEVICE)
    print(f"{name:28s} n={len(w):4d}  mean error {e.mean():.4f}  sd {e.std():.4f}")

normal, training file        n= 283  mean error 0.0322  sd 0.0210
normal, test file            n= 711  mean error 0.0257  sd 0.0202
anomaly windows, test file   n= 268  mean error 0.0155  sd 0.0108


## 4 — Short training with held-out normal windows

The model is trained for 20 epochs on the first 80% of P-3's normal training windows, in
time order. It is then checked on the held-out 20%, on normal test-file windows and on
anomaly windows, and scored point by point on the whole test file.

In [5]:
nw = d["normal_windows"]
cut = int(0.8 * len(nw))
train_w, held_w = nw[:cut], nw[cut:]
sc.seed_everything(42)
m = sc.LSTMAutoencoder(25).to(DEVICE)
opt = torch.optim.Adam(m.parameters(), lr=1e-3)
X = torch.as_tensor(train_w)
losses = []
for ep in range(20):
    m.train(); perm = torch.randperm(len(X)); tot = 0; nb = 0
    for i in range(0, len(X), 32):
        b = X[perm[i:i + 32]].to(DEVICE)
        opt.zero_grad(); l = torch.nn.functional.mse_loss(m(b), b); l.backward(); opt.step()
        tot += l.item(); nb += 1
    losses.append(tot / nb)
print(f"training loss: first epoch {losses[0]:.5f}, last epoch {losses[-1]:.5f}")
for name, w in {"held-out normal, training file": held_w, **{k: v for k, v in groups.items() if "test" in k}}.items():
    e = sc.window_errors(m, w, DEVICE)
    print(f"{name:32s} mean error {e.mean():.5f}")
pw, _ = sc.pointwise_scores(m, d["test"], DEVICE)
print(f"point-wise ROC-AUC on P-3's test file: {roc_auc_score(d['labels'], pw):.3f} "
      f"(anomalous fraction {d['labels'].mean():.3f})")

training loss: first epoch 0.03067, last epoch 0.01769
held-out normal, training file   mean error 0.01449
normal, test file                mean error 0.01235
anomaly windows, test file       mean error 0.00424


point-wise ROC-AUC on P-3's test file: 0.421 (anomalous fraction 0.157)


## 5 — Learning curve and one reconstruction

In [6]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(range(1, 21), losses, marker="o"); ax[0].set_xlabel("epoch"); ax[0].set_ylabel("MSE")
ax[0].set_title("training loss, P-3")
with torch.no_grad():
    r = m(torch.as_tensor(held_w[:1], device=DEVICE)).cpu().numpy()
ax[1].plot(held_w[0, :, 0], label="held-out window"); ax[1].plot(r[0, :, 0], "--", label="reconstruction")
ax[1].set_title("feature 0 of a held-out normal window"); ax[1].legend()
fig.tight_layout(); fig.savefig(os.path.join(OUT, "nb04_autoencoder_quick_test.png"), dpi=120); plt.close(fig)

## 6 — Save a reproducible untrained starting point

This file is an **untrained** random initialisation. It is useful as a fixed starting point.
It must never be used as a trained baseline.

In [7]:
sc.seed_everything(42)
init = sc.LSTMAutoencoder(25)
path = os.path.join(OUT, "lstm_ae_init_seed42.pt")
torch.save({"model_state_dict": init.state_dict(),
            "architecture": {"input_size": 25, "seq_len": 30, "hidden1": 64, "hidden2": 32, "latent": 16},
            "note": "UNTRAINED random initialisation, torch seed 42. Not a baseline.",
            "git_commit": sc.git_commit()}, path)
print("saved", path)

saved /home/user/Objective-2/Corrected-Notebooks/outputs/lstm_ae_init_seed42.pt
